# NAM: Neural Additive Model

NAM learns one neural shape function per encoded scalar column. Optional explicitly selected interactions extend the model without losing a term-wise decomposition.


## Model


$$
\eta(x)=\beta_0+\sum_{j=1}^{d}f_j(x_j)
          +\sum_{S\in\mathcal I}f_S(x_S).
$$

Each $f_j$ is an independent feature network. NAMpy also supports ExU and centered-ReLU first layers.


## Shared estimator API

All neural estimators use `fit`, `predict`, `score`, `evaluate`, and
`predict_components`. The component result reconstructs predictions on the link
scale and supports shared term-importance and plotting utilities. Constructor
options such as `numerical_preprocessing` and `categorical_preprocessing` are forwarded to
PreTab and are fitted on training rows only.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(7)
n = 180
X = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
    "group": rng.choice(["a", "b", "c"], size=n),
})
y = (
    np.sin(np.pi * X["x1"])
    + 0.35 * X["x2"] ** 2
    + 0.30 * (X["group"] == "b")
    + rng.normal(0.0, 0.12, n)
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Set True to run the small fit and all fitted-model demonstrations.
RUN_TRAINING = False


## Construct the estimator


In [ ]:
from nampy.models import NAMClassifier, NAMLSS, NAMRegressor


model = NAMRegressor(
    feature_layer="exu",
    layer_sizes=[32, 16],
    interactions=(("x1", "x2"),),
    output_regularization=1e-4,
    l2_regularization=1e-6,
    dropout=0.0,
)
model.get_params(deep=False)


## Fit and inspect

Enable `RUN_TRAINING` above for a short demonstration. Real work should use a
larger validation set, enough epochs, and early stopping.


In [ ]:
if RUN_TRAINING:
    model.fit(
        X_train,
        y_train,
        max_epochs=3,
        batch_size=64,
        random_state=7,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    predictions = model.predict(X_test)
    r2 = model.score(X_test, y_test)
    metrics = model.evaluate(X_test, y_test)
    components = model.predict_components(X_test, center=True)
    components.validate_additive_reconstruction()
    display({"R2": r2, **metrics})
    display(model.term_importance(X_test).head())


## Model-specific controls

Use `feature_layer`, adaptive widths, or `feature_widths` to control individual shape networks. `interactions` is preferable to generating every combination.


In [ ]:
model.set_params(
    adaptive_width=True,
    num_basis_functions=64,
    units_multiplier=2,
    feature_widths={"x1": 24},
)
if RUN_TRAINING:
    display(model.interaction_importance(X_test))
    model.plot_terms(X_test, pages=1)


## Task variants and limits

`NAMClassifier` adds probabilities and class labels. `NAMLSS(family='normal')` produces additive distribution-parameter predictors.
